# TinyStories SLM on Colab / Kaggle
All code lives in the repo's `.py` files; this notebook only runs commands. Pick a **GPU (T4)** runtime. Training uses **fp16 autocast + GradScaler** (the T4 has no fast bf16); add `--dtype fp32` if you ever see NaNs you cannot fix.

In [ ]:
# 1. get the code and install
!git clone https://github.com/InfernoAsura/SLM.git slm
%cd slm
!pip -q install -r requirements-dev.txt

In [ ]:
# 2. where to keep things that took time to make (survives a disconnect)
import os
WORK = '/kaggle/working' if os.path.exists('/kaggle') else '/content/drive/MyDrive/slm'
if WORK.startswith('/content/drive'):
    from google.colab import drive; drive.mount('/content/drive')
os.makedirs(f'{WORK}/data', exist_ok=True); print(WORK)

In [ ]:
# 3. download TinyStories V2 (GPT-4) train + valid
!wget -q -nc -P {WORK}/data https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-train.txt
!wget -q -nc -P {WORK}/data https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-valid.txt

In [ ]:
# 4. tests
!python -m pytest -q

In [ ]:
# 5. tokenizer (uses all CPU cores) and uint16 datasets
!python scripts/train_tokenizer.py --input {WORK}/data/TinyStoriesV2-GPT4-train.txt --out-dir {WORK}/tokenizer
!python scripts/encode_dataset.py --tokenizer-dir {WORK}/tokenizer --input {WORK}/data/TinyStoriesV2-GPT4-valid.txt --output {WORK}/data/valid.npy
!python scripts/encode_dataset.py --tokenizer-dir {WORK}/tokenizer --input {WORK}/data/TinyStoriesV2-GPT4-train.txt --output {WORK}/data/train.npy

In [ ]:
# 6. learning-rate pilots (300 steps each)
!python scripts/lr_sweep.py --train-data {WORK}/data/train.npy --val-data {WORK}/data/valid.npy --out-root {WORK}/runs/lr_sweep

In [ ]:
# 7. main run (add --resume {WORK}/runs/main/ckpt.pt after a disconnect)
!python train.py --train-data {WORK}/data/train.npy --val-data {WORK}/data/valid.npy --out-dir {WORK}/runs/main --max-steps 5000 --lr 3e-3 --compile

In [ ]:
# 8. release folder + a story on CPU
!mkdir -p release && cp {WORK}/runs/main/model.pt {WORK}/runs/main/config.json release/ && cp {WORK}/tokenizer/vocab.json {WORK}/tokenizer/merges.txt release/
!python inference.py --model-dir release --prompt "Once upon a time" --max-new-tokens 256 --temperature 0.8 --top-p 0.9 --seed 42 --device cpu